# QLoRA fine-tuning notebook for `Qwen/Qwen2.5-7B-Instruct`

This notebook trains the model with supervised fine-tuning.

> Replace `DATA_PATH` below with your CSV path.

In [1]:
# If you are in Colab/Kaggle, run this once.
# Restart the runtime after installation if the environment asks for it.

%pip install -q -U \
    "transformers>=4.37.0" \
    "accelerate>=0.30.0" \
    "datasets>=2.18.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "bitsandbytes>=0.43.0" \
    "pandas>=2.0.0" \
    "scikit-learn>=1.4.0" \
    "matplotlib>=3.8.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires panda

In [2]:
import os
import gc
import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH = "/content/final_pairs.csv"
OUTPUT_DIR = "./qwen25_7b_qlora_bias"
SEED = 42

# Sequence/training config
MAX_LENGTH = 1536
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 50

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook expects a CUDA GPU runtime. "
        "QLoRA with bitsandbytes 4-bit training is intended for GPU environments."
    )

DEVICE_NAME = torch.cuda.get_device_name(0)
BF16_SUPPORTED = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16

torch.backends.cuda.matmul.allow_tf32 = True

print(f"GPU: {DEVICE_NAME}")
print(f"BF16 supported: {BF16_SUPPORTED}")
print(f"Compute dtype: {COMPUTE_DTYPE}")

GPU: Tesla T4
BF16 supported: True
Compute dtype: torch.bfloat16


## Load CSV and split train/validation

In [4]:
df = pd.read_csv(DATA_PATH)

required_columns = {"question", "answer"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"CSV must contain columns {required_columns}. Missing: {missing}")

df = df[["question", "answer"]].copy()
df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df[(df["question"] != "") & (df["answer"] != "")]
df = df.drop_duplicates().reset_index(drop=True)

print(f"Total examples: {len(df):,}")
display(df.head(3))

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df):,}")
print(f"Val size:   {len(val_df):,}")

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

Total examples: 3,304


,question,answer
0,How is the operational effectiveness of the Ru...,The Russian Black Sea Fleet is portrayed as la...
1,What does Russia's attempt to develop naval dr...,Russia's efforts to copy Ukrainian naval drone...
2,"Why is the Black Sea described as a ""gray zone...","The Black Sea is considered a ""gray zone"" beca..."


Train size: 2,643
Val size:   661


## Tokenizer and chat formatting

We use the **Qwen chat template** so the model sees each example in the same format it expects at inference time.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pad_token: <|endoftext|>
eos_token: <|im_end|>


In [ ]:
def make_prompt(question: str) -> str:
    '''
    Prompt that ends right where the assistant answer should begin.
    Use add_generation_prompt=True to get the assistant prefix.
    '''
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def make_full_text(question: str, answer: str) -> str:
    prompt_text = make_prompt(question)
    # adding EOS so the model learns where the assistant answer ends.
    return prompt_text + answer + tokenizer.eos_token

In [13]:
make_prompt("asas")

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nasas<|im_end|>\n<|im_start|>assistant\n'

In [14]:
make_full_text("are they bad", "yeah")

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nare they bad<|im_end|>\n<|im_start|>assistant\nyeah<|im_end|>'

## Optional: baseline generation before fine-tuning

We generate one answer from the base model now, so later we can compare it against the fine-tuned model on the same question.

In [15]:
example_question = val_df.iloc[0]["question"]
print("Example question used for before/after comparison:\n")
print(example_question)

Example question used for before/after comparison:

How does the Ukrainian military maintain its defensive integrity against Russian attempts to disrupt logistics?


In [16]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0},
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [ ]:
@torch.no_grad()
def generate_answer(model, question: str, max_new_tokens: int = 180, temperature: float = 0.2):
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # remove prompt tokens so we only decode the new assistant answer.
    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


base_answer_before_ft = generate_answer(base_model, example_question)
print("=== Base model answer (before fine-tuning) ===")
print(base_answer_before_ft)

=== Base model answer (before fine-tuning) ===
The Ukrainian military faces significant challenges in maintaining its defensive integrity against Russian attempts to disrupt logistics, but there are several strategies and measures that have been employed to mitigate these disruptions:

1. **Enhanced Coordination with NATO and Allies**: Ukraine has sought closer cooperation with NATO and other Western allies to secure more advanced weaponry, intelligence sharing, and logistical support. This includes the provision of modern anti-tank missiles, drones, and other critical supplies.

2. **Improved Logistics Management**: Efforts to streamline and improve internal logistics within Ukraine's armed forces have been intensified. This includes better planning, coordination, and use of technology to manage supply chains and ensure that troops have the necessary equipment and resources.

3. **Use of Civilian Infrastructure**: Leveraging civilian infrastructure can help reduce reliance on military

## Preprocess dataset with explicit loss masking

We tokenize:

- the **prompt only**
- the **full prompt + answer**

In [18]:
IGNORE_INDEX = -100

def preprocess_batch(batch):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for question, answer in zip(batch["question"], batch["answer"]):
        prompt_text = make_prompt(str(question).strip())
        full_text = prompt_text + str(answer).strip() + tokenizer.eos_token

        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False,
        )["input_ids"]

        full_ids = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_LENGTH,
        )["input_ids"]

        prompt_len = min(len(prompt_ids), len(full_ids))
        labels = [IGNORE_INDEX] * prompt_len + full_ids[prompt_len:]

        all_input_ids.append(full_ids)
        all_attention_masks.append([1] * len(full_ids))
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }


train_tokenized = train_ds.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train set",
)

val_tokenized = val_ds.map(
    preprocess_batch,
    batched=True,
    remove_columns=val_ds.column_names,
    desc="Tokenizing val set",
)

print(train_tokenized)
print(val_tokenized)

Tokenizing train set:   0%|          | 0/2643 [00:00<?, ? examples/s]

Tokenizing val set:   0%|          | 0/661 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2643
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 661
})


In [ ]:
# verifying that labels only contain the assistant answer tokens.
sample = train_tokenized[0]

decoded_full = tokenizer.decode(sample["input_ids"], skip_special_tokens=False)
decoded_loss_part = tokenizer.decode(
    [tok for tok in sample["labels"] if tok != IGNORE_INDEX],
    skip_special_tokens=False,
)

print("=== Full tokenized text ===")
print(decoded_full[:2000])

print("\n=== Tokens that contribute to loss ===")
print(decoded_loss_part[:1200])

=== Full tokenized text ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Why is international cooperation in training AI models for defense framed as a strategic necessity?<|im_end|>
<|im_start|>assistant
Collaborative access to real-world battlefield data allows Ukraine and its partners to modernize defense technologies and accelerate the development of autonomous systems capable of rapid target detection and situational analysis.<|im_end|>

=== Tokens that contribute to loss ===
Collaborative access to real-world battlefield data allows Ukraine and its partners to modernize defense technologies and accelerate the development of autonomous systems capable of rapid target detection and situational analysis.<|im_end|>


In [20]:
@dataclass
class CausalLMDataCollator:
    tokenizer: AutoTokenizer
    label_pad_token_id: int = IGNORE_INDEX
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        if self.pad_to_multiple_of is not None and max_len % self.pad_to_multiple_of != 0:
            max_len = ((max_len // self.pad_to_multiple_of) + 1) * self.pad_to_multiple_of

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            seq_len = len(f["input_ids"])
            pad_len = max_len - seq_len

            input_ids.append(f["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [self.label_pad_token_id] * pad_len)

        batch = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }
        return batch


data_collator = CausalLMDataCollator(tokenizer=tokenizer)

## Attach QLoRA adapters

We keep the 4-bit base model frozen and train LoRA adapters.

`target_modules` below includes both:
- attention projections
- MLP projections

That gives the adapters more capacity than attention-only LoRA.

In [21]:
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## Define the supervised fine-tuning training objective

This is ordinary causal language-model training, but because `labels` are masked for prompt tokens,
the loss is computed **only over answer tokens**.

In [25]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",

    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    bf16=BF16_SUPPORTED,
    fp16=not BF16_SUPPORTED,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,

    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

## Train

In [ ]:
train_result = trainer.train()

print("\n=== Final train result ===")
print(train_result)

In [ ]:
eval_metrics = trainer.evaluate()
eval_metrics["perplexity"] = math.exp(eval_metrics["eval_loss"]) if eval_metrics["eval_loss"] < 20 else float("inf")

print("=== Eval metrics ===")
for k, v in eval_metrics.items():
    print(f"{k}: {v}")

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved adapter checkpoint to: {OUTPUT_DIR}")

## Visualize training logs

In [ ]:
logs_df = pd.DataFrame(trainer.state.log_history)
display(logs_df.tail(20))

if "loss" in logs_df.columns:
    plt.figure(figsize=(8, 4))
    train_logs = logs_df.dropna(subset=["loss"])
    plt.plot(train_logs["step"], train_logs["loss"], label="train_loss")
    if "eval_loss" in logs_df.columns:
        eval_logs = logs_df.dropna(subset=["eval_loss"])
        plt.plot(eval_logs["step"], eval_logs["eval_loss"], label="eval_loss")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("Training / evaluation loss")
    plt.legend()
    plt.show()

## Compare base model vs fine-tuned model

In [ ]:
fine_tuned_answer = generate_answer(model, example_question)

print("=== Question ===")
print(example_question)

print("\n=== Base model answer (before fine-tuning) ===")
print(base_answer_before_ft)

print("\n=== Fine-tuned model answer ===")
print(fine_tuned_answer)

## Optional: test on your own prompt

In [ ]:
custom_question = "Put your own question here"

if custom_question.strip():
    print("=== Fine-tuned model answer ===")
    print(generate_answer(model, custom_question))